# Transformers — Exercises

Companion to the note [Transformers](Transformers.md).

Practise scaled dot-product attention, masking, multi-head attention, positional encodings (sinusoidal and RoPE), LayerNorm and the residual block structure, plus the arithmetic of parameter counts, attention cost and the KV-cache. Everything is built from scratch in NumPy and checked with SciPy references and property tests.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×5 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import softmax as sp_softmax
rng = np.random.default_rng(2)

def num_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar function f() w.r.t. array x (modified in place, restored)."""
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Why positional information is needed
*🧠 Concept · ★☆☆*

Show that self-attention without positional encodings is **permutation-equivariant**: if the rows of $X$ are permuted by $P$,
then $\mathrm{SelfAttn}(PX) = P\,\mathrm{SelfAttn}(X)$. What does this imply for "dog bites man" vs "man bites dog"?
Name the four options the note lists for injecting positions.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

With $Q = XW_Q$ etc., $(PX)W_Q = P(XW_Q)$ and $\mathrm{softmax}(PSP^\top) = P\,\mathrm{softmax}(S)\,P^\top$ (row-wise softmax).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{softmax}\big(PQ(PK)^\top/\sqrt{d_k}\big)PV = P\,\mathrm{softmax}(QK^\top/\sqrt{d_k})P^\top P V = P\,\mathrm{Attn}(Q,K,V)$ since $P^\top P = I$.
So the model sees a **bag of tokens**: both sentences produce the same multiset of outputs. Positions are injected with
**sinusoidal**, **learned** absolute embeddings, **RoPE** (rotations of $q,k$) or **ALiBi** (a distance-proportional bias on the scores).
We verify equivariance numerically in Exercise 12.

</details>

### Exercise 2 · Encoder, decoder, encoder–decoder
*🧠 Concept · ★☆☆*

For each family in the note's table (BERT, GPT, T5, ViT) state: (a) which attention mask is used, (b) the training objective, (c) a typical task.
Why can BERT not be used directly to generate text left-to-right?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Masked LM sees both sides; next-token prediction must not see the future.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| Model | Mask | Objective | Typical use |
|---|---|---|---|
| BERT (encoder) | none (bidirectional) | masked LM: predict ~15% masked tokens | classification, NER, retrieval embeddings |
| GPT/Llama (decoder) | **causal** | next-token prediction | generation, chat |
| T5 (enc–dec) | encoder none, decoder causal + cross-attention | span corruption, seq2seq | translation, summarisation |
| ViT (encoder) | none | supervised classification (or MAE) | image classification |

BERT was trained with bidirectional context, so its predictions at position $t$ rely on tokens to the right; there is no
calibrated $p(x_t\mid x_{<t})$ to sample from.

</details>

### Exercise 3 · Training recipe: pre-LN, warm-up, clipping
*🧠 Concept · ★★☆*

The note recommends *pre-LN*, AdamW with warm-up and cosine decay, and gradient clipping.

1. Write the post-LN and pre-LN versions of a sublayer and explain why pre-LN trains more stably in deep stacks.
2. Why is learning-rate warm-up especially important for Adam-type optimisers early in training?
3. What does the KV-cache store during generation, and why does it make each new token $O(n)$ instead of $O(n^2)$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Pre-LN keeps an un-normalised identity path from input to output. Adam's second-moment estimate is noisy in the first steps.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Post-LN: $x \leftarrow \mathrm{LN}(x + F(x))$. Pre-LN: $x \leftarrow x + F(\mathrm{LN}(x))$. In pre-LN the residual stream is a pure sum,
   so there is a clean identity gradient path (as in [[CNN|ResNet]]); in post-LN every LayerNorm sits on the path and gradients near the
   output are large, requiring careful warm-up.
2. Adam divides by $\sqrt{\hat v_t}$, which is estimated from very few samples early on; tiny $\hat v$ produce huge, erratic steps.
   Warm-up keeps steps small until the statistics settle ([[Optimizers]]).
3. It stores the keys and values of all previous tokens for every layer. A new token only computes its own $q,k,v$ and attends to the
   cached $K,V$: $O(nd)$ per token instead of recomputing all $n^2$ interactions. The price is memory (Exercise 8).

</details>

## Part B · By hand

### Exercise 4 · Attention by hand
*✍️ By hand · ★☆☆*

Let $Q = \begin{pmatrix}1&0\\0&1\end{pmatrix}$, $K = \begin{pmatrix}1&0\\1&1\end{pmatrix}$, $V = \begin{pmatrix}1&2\\3&4\end{pmatrix}$, $d_k = 2$.
Compute the attention weights $A = \mathrm{softmax}(QK^\top/\sqrt{d_k})$ and the output $AV$ (3 decimals).

In [ ]:
Q_h = np.array([[1, 0], [0, 1.]]); K_h = np.array([[1, 0], [1, 1.]]); V_h = np.array([[1, 2], [3, 4.]])
A_hand = None    # 2x2
out_hand = None  # 2x2

_A = sp_softmax(Q_h @ K_h.T / np.sqrt(2), axis=1)
check('attention weights', A_hand, _A, tol=1e-3); check('output', out_hand, _A @ V_h, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$QK^\top = \begin{pmatrix}1&1\\0&1\end{pmatrix}$. Row 1 has equal scores.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Row 2: scores $(0, 1/\sqrt2) = (0, 0.7071)$; $\mathrm{softmax} = (1, e^{0.7071})/(1+e^{0.7071})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$QK^\top/\sqrt2 = \begin{pmatrix}0.707&0.707\\0&0.707\end{pmatrix}$. Row 1: $(0.5, 0.5)$. Row 2: $e^{0.7071} = 2.028$, so $(0.330, 0.670)$.
$AV$: row 1 $= 0.5(1,2)+0.5(3,4) = (2, 3)$; row 2 $= 0.330(1,2) + 0.670(3,4) = (2.340, 3.340)$.

```python
Q_h = np.array([[1, 0], [0, 1.]]); K_h = np.array([[1, 0], [1, 1.]]); V_h = np.array([[1, 2], [3, 4.]])
A_hand = [[0.5, 0.5], [0.3302, 0.6698]]
out_hand = [[2.0, 3.0], [2.3395, 3.3395]]
```

</details>

### Exercise 5 · Why divide by √d_k?
*✍️ By hand · ★☆☆*

Assume $q, k \in \mathbb R^{d_k}$ have i.i.d. components with mean 0 and variance 1, independent of each other.
Show $\mathrm{Var}(q^\top k) = d_k$. For $d_k = 64$, store the variance of the raw score and of the scaled score $q^\top k/\sqrt{d_k}$.

In [ ]:
var_raw = None
var_scaled = None

_q, _k = rng.normal(size=(200000, 64)), rng.normal(size=(200000, 64))
_s = np.sum(_q * _k, axis=1)
check('Var(q.k), simulated', var_raw, _s.var(), tol=1.5); check('Var(q.k/sqrt(d)), simulated', var_scaled, (_s / 8).var(), tol=0.03)

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{Var}(q_ik_i) = \mathbb E[q_i^2]\mathbb E[k_i^2] - 0 = 1$, and the $d_k$ terms are independent.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E[q_ik_i] = 0$, $\mathrm{Var}(q_ik_i) = \mathbb E[q_i^2]\mathbb E[k_i^2] = 1$, sum of $d_k$ independent terms: $\mathrm{Var}(q^\top k) = d_k = 64$
(std 8). Scaling by $1/\sqrt{d_k}$ gives variance 1. Without it the softmax saturates (one weight ≈ 1) and its gradients vanish (Exercise 10).

```python
var_raw = 64
var_scaled = 1
```

</details>

### Exercise 6 · Parameter count of one block
*✍️ By hand · ★★☆*

An encoder block with $d_{model}=512$, $h=8$ heads, $d_{ff}=2048$ has: $W_Q, W_K, W_V, W_O \in \mathbb R^{512\times512}$ (each with bias),
an MLP $512\to2048\to512$ (with biases), and two LayerNorms (each with $\gamma, \beta$). Count all parameters.
Does the number of heads change the count?

In [ ]:
params_block = None

_d, _f = 512, 2048
_lin = lambda i, o: i * o + o
check('parameters', params_block, 4 * _lin(_d, _d) + _lin(_d, _f) + _lin(_f, _d) + 2 * 2 * _d)

<details>
<summary><b>💡 Hint</b></summary>

Attention: $4(d^2+d)$. MLP: $(d\cdot d_{ff}+d_{ff}) + (d_{ff}\cdot d + d)$. LayerNorms: $2\cdot 2d$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Attention $4(262{,}144 + 512) = 1{,}050{,}624$; MLP $1{,}050{,}624 + 1{,}049{,}088 = 2{,}099{,}712$; LayerNorm $2048$.
Total $\mathbf{3{,}152{,}384}$, about 2/3 in the MLP. The number of heads does **not** change it: heads split $d_{model}$ into $h$ slices of size $d_k = d/h$.

```python
params_block = 3152384
```

</details>

### Exercise 7 · Where does the compute go?
*✍️ By hand · ★★☆*

Count multiply-adds for one self-attention layer on $n$ tokens with width $d$ (single head is enough):
the projections $XW_Q, XW_K, XW_V$ cost $3nd^2$, and $QK^\top$ costs $n^2 d$. For $n=4096, d=1024$ compute the ratio
$\frac{n^2d}{3nd^2}$, and the sequence length at which the two are equal for $d=1024$.

In [ ]:
ratio_attn_proj = None
n_equal = None

check('ratio', ratio_attn_proj, 4096**2 * 1024 / (3 * 4096 * 1024**2))
check('break-even n', n_equal, 3 * 1024)

<details>
<summary><b>💡 Hint</b></summary>

The ratio simplifies to $n/(3d)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{n^2d}{3nd^2} = \frac{n}{3d} = \frac{4096}{3072} \approx 1.333$. Equal when $n = 3d = 3072$.
For short sequences the projections and MLP dominate; the $O(n^2d)$ term only takes over for long contexts, which is what
FlashAttention and sparse/linear attention target.

```python
ratio_attn_proj = 4096 / 3072
n_equal = 3072
```

</details>

### Exercise 8 · KV-cache memory
*✍️ By hand · ★★☆*

A decoder with $L=32$ layers and $d_{model}=4096$ (full multi-head attention, no grouping) generates with a context of $n=2048$
tokens in fp16 (2 bytes). How much memory does the KV-cache take for **one** sequence, in GiB? How many bytes per token?

In [ ]:
kv_gib = None
kv_bytes_per_token = None

check('KV cache (GiB)', kv_gib, 2 * 32 * 2048 * 4096 * 2 / 2**30)
check('bytes per token', kv_bytes_per_token, 2 * 32 * 4096 * 2)

<details>
<summary><b>💡 Hint</b></summary>

Per layer and token you store one key and one value vector of size $d_{model}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$2\,(K,V) \times 32 \times 2048 \times 4096 \times 2\text{ B} = 2^{30}\text{ B} = 1\text{ GiB}$; per token $2\cdot32\cdot4096\cdot2 = 524{,}288$ B (0.5 MiB).
A batch of 32 such sequences needs 32 GiB, which is why grouped-query attention and cache quantisation exist.

```python
kv_gib = 1.0
kv_bytes_per_token = 524288
```

</details>

## Part C · Code from scratch

### Exercise 9 · Stable softmax and scaled dot-product attention
*💻 Code · ★☆☆*

Implement `softmax(S, axis=-1)` that does not overflow for large inputs, and
`attention(Q, K, V, mask=None)` returning `(output, weights)`. `mask` is a boolean array, `True` where attention is
**allowed** (you will use it in Exercise 11; masked scores become $-\infty$).

In [ ]:
def softmax(S, axis=-1):
    return None

def attention(Q, K, V, mask=None):
    return None, None

big = softmax(np.array([1000., 1001., 1002.]))
Qa, Ka, Va = rng.normal(size=(5, 8)), rng.normal(size=(7, 8)), rng.normal(size=(7, 3))
out_a, A_a = attention(Qa, Ka, Va)

check('no overflow', big, sp_softmax([1000., 1001., 1002.]))
check('weights', A_a, sp_softmax(Qa @ Ka.T / np.sqrt(8), axis=1))
check('output', out_a, sp_softmax(Qa @ Ka.T / np.sqrt(8), axis=1) @ Va)

<details>
<summary><b>💡 Hint</b></summary>

Subtract `S.max(axis, keepdims=True)` before exponentiating; softmax is invariant to adding a constant per row.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Shifting by the row max makes the largest exponent $e^0=1$, so nothing overflows; the result is mathematically identical.

```python
def softmax(S, axis=-1):
    S = S - S.max(axis=axis, keepdims=True)
    E = np.exp(S)
    return E / E.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    S = Q @ K.T / np.sqrt(Q.shape[-1])
    if mask is not None:
        S = np.where(mask, S, -np.inf)
    A = softmax(S, axis=-1)
    return A @ V, A

big = softmax(np.array([1000., 1001., 1002.]))
Qa, Ka, Va = rng.normal(size=(5, 8)), rng.normal(size=(7, 8)), rng.normal(size=(7, 3))
out_a, A_a = attention(Qa, Ka, Va)
```

</details>

### Exercise 10 · Softmax saturation without scaling
*💻 Code · ★☆☆*

For $d \in \{4, 64, 512\}$ draw $Q, K$ with $32$ rows of i.i.d. $\mathcal N(0,1)$ entries. Compute the average (over rows) of the
largest attention weight, once with scores $QK^\top$ and once with $QK^\top/\sqrt d$. Store dicts `maxw_raw[d]` and `maxw_scaled[d]`.

In [ ]:
maxw_raw = None
maxw_scaled = None

check('unscaled, d=512: nearly one-hot (> 0.85)', None if maxw_raw is None else maxw_raw[512] > 0.85, True)
check('scaled, d=512: spread out (< 0.4)', None if maxw_scaled is None else maxw_scaled[512] < 0.4, True)
check('scaled weights barely depend on d', None if maxw_scaled is None else abs(maxw_scaled[512] - maxw_scaled[4]) < 0.15, True)

<details>
<summary><b>💡 Hint</b></summary>

`softmax(Q @ K.T).max(axis=1).mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Unscaled scores have std $\sqrt d \approx 22.6$ at $d=512$: the softmax is essentially an argmax, its Jacobian $\mathrm{diag}(a)-aa^\top$ is ≈ 0,
so gradients vanish. With $1/\sqrt d$ the score scale is independent of $d$.

```python
maxw_raw, maxw_scaled = {}, {}
for d in (4, 64, 512):
    Q, K = rng.normal(size=(32, d)), rng.normal(size=(32, d))
    maxw_raw[d] = softmax(Q @ K.T).max(axis=1).mean()
    maxw_scaled[d] = softmax(Q @ K.T / np.sqrt(d)).max(axis=1).mean()
print(maxw_raw, maxw_scaled)
```

</details>

### Exercise 11 · Causal masking
*💻 Code · ★★☆*

Implement `causal_mask(n)` (boolean, `True` on and below the diagonal). Use it with your `attention` for self-attention
($Q=K=V=X$) and verify the defining property of a decoder: changing tokens $t\ge 3$ must not change outputs at positions $0,1,2$.
Store the max abs change of the first three outputs in `leak` (should be 0) and of all outputs in `change_all` (should be > 0).

In [ ]:
def causal_mask(n):
    return None

X_c = rng.normal(size=(6, 4))
X_c2 = X_c.copy(); X_c2[3:] = rng.normal(size=(3, 4))
leak = None
change_all = None

check('mask = lower triangle', causal_mask(6), np.tril(np.ones((6, 6), bool)))
check('no information from the future', None if leak is None else leak < 1e-12, True)
check('later positions do change', None if change_all is None else change_all > 1e-3, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.tril(np.ones((n, n), dtype=bool))`. Row $t$ may attend to columns $\le t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Because masked scores are $-\infty$, their softmax weight is exactly $0$, so position $t$ is a function of $x_{\le t}$ only.
This is what lets a decoder be trained on all positions in parallel with next-token prediction (teacher forcing).

```python
def causal_mask(n):
    return np.tril(np.ones((n, n), dtype=bool))

X_c = rng.normal(size=(6, 4))
X_c2 = X_c.copy(); X_c2[3:] = rng.normal(size=(3, 4))
o1, _ = attention(X_c, X_c, X_c, causal_mask(6))
o2, _ = attention(X_c2, X_c2, X_c2, causal_mask(6))
leak = np.abs(o1[:3] - o2[:3]).max()
change_all = np.abs(o1 - o2).max()
```

</details>

### Exercise 12 · Multi-head attention
*💻 Code · ★★☆*

Implement `mha(X, Wq, Wk, Wv, Wo, h)` for self-attention: project with $W_Q, W_K, W_V\in\mathbb R^{d\times d}$, split the $d$ columns
into $h$ heads of size $d/h$, run `attention` per head, concatenate, multiply by $W_O$. Then verify permutation equivariance (Exercise 1).

In [ ]:
def mha(X, Wq, Wk, Wv, Wo, h):
    return None

n_m, d_m = 5, 8
X_m = rng.normal(size=(n_m, d_m))
Wq_m, Wk_m, Wv_m, Wo_m = (rng.normal(0, d_m ** -0.5, size=(d_m, d_m)) for _ in range(4))
Y_m = mha(X_m, Wq_m, Wk_m, Wv_m, Wo_m, h=2)
perm = rng.permutation(n_m)
Y_perm = mha(X_m[perm], Wq_m, Wk_m, Wv_m, Wo_m, h=2)

_Q, _K, _V = X_m @ Wq_m, X_m @ Wk_m, X_m @ Wv_m
_heads = [sp_softmax(_Q[:, s] @ _K[:, s].T / 2, axis=1) @ _V[:, s] for s in (slice(0, 4), slice(4, 8))]
check('2-head output', Y_m, np.concatenate(_heads, axis=1) @ Wo_m)
check('permutation equivariance', Y_perm, None if Y_m is None else Y_m[perm])

<details>
<summary><b>💡 Hint</b></summary>

Head $j$ uses columns `j*dk:(j+1)*dk` of $Q, K, V$, with $d_k = d/h$ (so the scaling is $1/\sqrt{d/h}$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each head computes its own attention pattern in a $d/h$-dimensional subspace (e.g. one head for syntax, one for coreference); $W_O$ mixes them.
Cost and parameter count are the same as a single head of width $d$.

```python
def mha(X, Wq, Wk, Wv, Wo, h):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    dk = Q.shape[1] // h
    heads = [attention(Q[:, j*dk:(j+1)*dk], K[:, j*dk:(j+1)*dk], V[:, j*dk:(j+1)*dk])[0] for j in range(h)]
    return np.concatenate(heads, axis=1) @ Wo

n_m, d_m = 5, 8
X_m = rng.normal(size=(n_m, d_m))
Wq_m, Wk_m, Wv_m, Wo_m = (rng.normal(0, d_m ** -0.5, size=(d_m, d_m)) for _ in range(4))
Y_m = mha(X_m, Wq_m, Wk_m, Wv_m, Wo_m, h=2)
perm = rng.permutation(n_m)
Y_perm = mha(X_m[perm], Wq_m, Wk_m, Wv_m, Wo_m, h=2)
```

</details>

### Exercise 13 · Sinusoidal positional encoding
*💻 Code · ★★☆*

Implement `sinusoidal_pe(n, d)` with $PE_{p,2i} = \sin(p/10000^{2i/d})$ and $PE_{p,2i+1} = \cos(p/10000^{2i/d})$.
Then verify two properties: every row has norm $\sqrt{d/2}$, and $PE_p^\top PE_{p+k}$ depends only on the offset $k$ (not on $p$).
Why does the second property help the model attend by *relative* position?

In [ ]:
def sinusoidal_pe(n, d):
    return None

PE = sinusoidal_pe(50, 16)

check('PE[1,0] = sin(1), PE[1,1] = cos(1)', None if PE is None else PE[1, :2], [np.sin(1), np.cos(1)])
check('row norms = sqrt(d/2)', None if PE is None else np.linalg.norm(PE, axis=1), np.full(50, np.sqrt(8)))
check('PE_p . PE_(p+5) independent of p', None if PE is None else np.ptp([PE[p] @ PE[p + 5] for p in range(40)]), 0.0, tol=1e-9)

<details>
<summary><b>💡 Hint</b></summary>

`freqs = 10000 ** (-np.arange(0, d, 2) / d)`; `angles = np.arange(n)[:, None] * freqs`; interleave sin and cos.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each pair is $(\sin p\omega_i, \cos p\omega_i)$, a unit vector, so the norm is $\sqrt{d/2}$.
$PE_p^\top PE_{p+k} = \sum_i [\sin p\omega_i \sin (p+k)\omega_i + \cos p\omega_i\cos(p+k)\omega_i] = \sum_i \cos k\omega_i$, independent of $p$.
More strongly, $PE_{p+k}$ is a fixed rotation of $PE_p$, so a linear map can express "look $k$ tokens back". RoPE builds this rotation directly into $q$ and $k$ (next exercise).

```python
def sinusoidal_pe(n, d):
    freqs = 10000.0 ** (-np.arange(0, d, 2) / d)
    ang = np.arange(n)[:, None] * freqs[None, :]
    pe = np.zeros((n, d))
    pe[:, 0::2], pe[:, 1::2] = np.sin(ang), np.cos(ang)
    return pe

PE = sinusoidal_pe(50, 16)
```

</details>

### Exercise 14 · Rotary position embedding (RoPE)
*💻 Code · ★★★*

RoPE rotates each pair $(x_{2i}, x_{2i+1})$ of a query/key at position $m$ by the angle $m\theta_i$, $\theta_i = 10000^{-2i/d}$.
Implement `rope(x, m)` for a single vector. Show numerically that (a) norms are preserved and (b)
$\mathrm{rope}(q, m)^\top \mathrm{rope}(k, n)$ depends only on $m-n$. Prove (b) in one line.

In [ ]:
def rope(x, m):
    return None

q_r, k_r = rng.normal(size=16), rng.normal(size=16)
s_a = None if rope(q_r, 0) is None else rope(q_r, 7) @ rope(k_r, 3)
s_b = None if rope(q_r, 0) is None else rope(q_r, 104) @ rope(k_r, 100)

check('norm preserved', None if rope(q_r, 5) is None else np.linalg.norm(rope(q_r, 5)), np.linalg.norm(q_r))
check('score depends only on m - n', s_a, s_b)
_th = 10000.0 ** (-np.arange(0, 16, 2) / 16) * 4
_z = (q_r[0::2] + 1j * q_r[1::2]) * np.exp(1j * _th)    # complex-number reference: rotation = multiplication by e^{i m theta}
check('matches complex rotation', None if rope(q_r, 4) is None else np.r_[rope(q_r, 4)[0::2], rope(q_r, 4)[1::2]], np.r_[_z.real, _z.imag])

<details>
<summary><b>💡 Hint 1</b></summary>

For pair $i$: $(x_{2i}\cos\phi - x_{2i+1}\sin\phi,\ x_{2i}\sin\phi + x_{2i+1}\cos\phi)$ with $\phi = m\theta_i$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For the proof write $R_m$ for the block-diagonal rotation matrix and use $R_m^\top R_n = R_{n-m}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{rope}(q,m)^\top\mathrm{rope}(k,n) = q^\top R_m^\top R_n k = q^\top R_{n-m} k$: a function of the offset only. So RoPE gives
**relative** position awareness inside the dot product, with no extra parameters, and it is what Llama-family models use.

```python
def rope(x, m):
    d = len(x)
    phi = m * 10000.0 ** (-np.arange(0, d, 2) / d)
    x1, x2 = x[0::2], x[1::2]
    out = np.empty_like(x, dtype=float)
    out[0::2] = x1 * np.cos(phi) - x2 * np.sin(phi)
    out[1::2] = x1 * np.sin(phi) + x2 * np.cos(phi)
    return out

q_r, k_r = rng.normal(size=16), rng.normal(size=16)
s_a = rope(q_r, 7) @ rope(k_r, 3)
s_b = rope(q_r, 104) @ rope(k_r, 100)
```

</details>

### Exercise 15 · LayerNorm forward and backward
*💻 Code · ★★★*

LayerNorm normalises each **row** (token): $\hat x = (x-\mu)/\sqrt{\sigma^2+\epsilon}$, $y = \gamma\odot\hat x + \beta$.
Implement `layernorm(X, gamma, beta, eps=1e-5)` and `layernorm_backward(X, gamma, G, eps=1e-5)` returning `dX`
for upstream gradient `G`. Check with finite differences.

In [ ]:
def layernorm(X, gamma, beta, eps=1e-5):
    return None

def layernorm_backward(X, gamma, G, eps=1e-5):
    return None

X_ln = rng.normal(2, 3, size=(4, 6)); g_ln = rng.normal(size=6); b_ln = rng.normal(size=6); G_ln = rng.normal(size=(4, 6))
Y_ln = layernorm(X_ln, np.ones(6), np.zeros(6))
dX_ln = layernorm_backward(X_ln, g_ln, G_ln)

check('rows have mean 0', None if Y_ln is None else Y_ln.mean(1), np.zeros(4))
check('rows have variance ~1', None if Y_ln is None else Y_ln.var(1), np.ones(4), tol=1e-4)
def _f():
    mu, var = X_ln.mean(1, keepdims=True), X_ln.var(1, keepdims=True)
    return np.sum(((X_ln - mu) / np.sqrt(var + 1e-5) * g_ln + b_ln) * G_ln)
check('dX (finite differences)', dX_ln, num_grad(_f, X_ln), tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

Let $g = G\odot\gamma$ (gradient w.r.t. $\hat x$) and $s = 1/\sqrt{\sigma^2+\epsilon}$ per row.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$dX = s\,\big(g - \overline{g} - \hat x\,\overline{g\odot\hat x}\big)$, where the bars are row means.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $g = G\odot\gamma$: $\frac{\partial L}{\partial x} = \frac{1}{\sqrt{\sigma^2+\epsilon}}\Big(g - \tfrac1d\sum_j g_j - \hat x\,\tfrac1d\sum_j g_j\hat x_j\Big)$.
The two subtracted terms project out the directions that normalisation removes (shifts and rescalings of the row).
Unlike BatchNorm, LayerNorm does not depend on the batch, so it behaves identically at train and test time and works for any sequence length.

```python
def layernorm(X, gamma, beta, eps=1e-5):
    mu, var = X.mean(-1, keepdims=True), X.var(-1, keepdims=True)
    return (X - mu) / np.sqrt(var + eps) * gamma + beta

def layernorm_backward(X, gamma, G, eps=1e-5):
    mu, var = X.mean(-1, keepdims=True), X.var(-1, keepdims=True)
    s = 1 / np.sqrt(var + eps)
    xhat = (X - mu) * s
    g = G * gamma
    return s * (g - g.mean(-1, keepdims=True) - xhat * (g * xhat).mean(-1, keepdims=True))

X_ln = rng.normal(2, 3, size=(4, 6)); g_ln = rng.normal(size=6); b_ln = rng.normal(size=6); G_ln = rng.normal(size=(4, 6))
Y_ln = layernorm(X_ln, np.ones(6), np.zeros(6))
dX_ln = layernorm_backward(X_ln, g_ln, G_ln)
```

</details>

### Exercise 16 · A full pre-LN encoder block
*💻 Code · ★★☆*

Assemble the block from the note (pre-LN variant):
$Z = X + \mathrm{MHA}(\mathrm{LN}(X))$, then $Y = Z + W_2\,\mathrm{ReLU}(W_1\,\mathrm{LN}(Z) + b_1) + b_2$ (row-wise).
`p` is a dict with `Wq, Wk, Wv, Wo, W1, b1, W2, b2` (use `X @ W1` conventions, $\gamma=1,\beta=0$). Check two properties:
with all weights zero the block is the identity, and it is permutation-equivariant.

In [ ]:
def encoder_block(X, p, h=2):
    return None

d_e, f_e = 8, 32
p_e = {k: rng.normal(0, 0.3, size=s) for k, s in
       dict(Wq=(d_e, d_e), Wk=(d_e, d_e), Wv=(d_e, d_e), Wo=(d_e, d_e), W1=(d_e, f_e), b1=(f_e,), W2=(f_e, d_e), b2=(d_e,)).items()}
p_zero = {k: np.zeros_like(v) for k, v in p_e.items()}
X_e = rng.normal(size=(6, d_e)); perm_e = rng.permutation(6)
Y_e = encoder_block(X_e, p_e)
Y_e_perm = encoder_block(X_e[perm_e], p_e)
Y_e_zero = encoder_block(X_e, p_zero)

check('zero weights -> identity (residual path)', Y_e_zero, X_e)
check('permutation equivariance', Y_e_perm, None if Y_e is None else Y_e[perm_e])
check('output shape', None if Y_e is None else Y_e.shape, (6, 8))

<details>
<summary><b>💡 Hint</b></summary>

Reuse `layernorm(X, np.ones(d), np.zeros(d))` and `mha` from the previous exercises.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With zero weights both sublayers output 0 and the residual stream passes $X$ through: deep pre-LN stacks start close to the identity,
which is a big part of why they train stably. Everything except attention acts row-wise, so the block inherits attention's permutation equivariance;
only positional encodings break it.

```python
def encoder_block(X, p, h=2):
    d = X.shape[1]
    ln = lambda A: layernorm(A, np.ones(d), np.zeros(d))
    Z = X + mha(ln(X), p['Wq'], p['Wk'], p['Wv'], p['Wo'], h)
    return Z + np.maximum(0, ln(Z) @ p['W1'] + p['b1']) @ p['W2'] + p['b2']

d_e, f_e = 8, 32
p_e = {k: rng.normal(0, 0.3, size=s) for k, s in
       dict(Wq=(d_e, d_e), Wk=(d_e, d_e), Wv=(d_e, d_e), Wo=(d_e, d_e), W1=(d_e, f_e), b1=(f_e,), W2=(f_e, d_e), b2=(d_e,)).items()}
p_zero = {k: np.zeros_like(v) for k, v in p_e.items()}
X_e = rng.normal(size=(6, d_e)); perm_e = rng.permutation(6)
Y_e = encoder_block(X_e, p_e)
Y_e_perm = encoder_block(X_e[perm_e], p_e)
Y_e_zero = encoder_block(X_e, p_zero)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Transformers](Transformers.md).*